<a href="https://colab.research.google.com/github/temahm/AiCon/blob/main/TIR_Gemini_MultiAgent_XGBoost_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TIR Multi-Agent Demo in Google Colab — XGBoost Version

This notebook demonstrates a **three-agent socio-technical AI workflow** with a **XGBoost prediction model**:

1. **Model 1 — Prediction Agent (XGBoost)**  
   Learns approval patterns from a synthetic training dataset and predicts approve/deny outcomes.

2. **Model 2 — TIR Interrogator (Gemini)**  
   Questions the prediction, detects proxy-risk concerns, highlights uncertainty, and surfaces missing context.

3. **Model 3 — Governance Agent (Gemini)**  
   Reviews the prediction plus the interrogator's findings and decides whether to **APPROVE, DENY, REVIEW, or ESCALATE**.

## This demo is similar to a realistic enterprise workflow:
- a trained ML model
- explainability with SHAP (SHapley Additive exPlanations)
- a separate interrogation layer
- a separate governance layer

##**Accountable decision systems need interrogation and governance combined with Accurate models.**


## 1) Install dependencies

In [25]:
!pip install -q google-genai matplotlib pandas==2.2.2 scikit-learn xgboost shap


## 2) Gemini setup

This notebook expects your Gemini API key to be stored in **Colab Secrets** as:

`GEMINI_API_KEY`


In [26]:
from google.colab import userdata
from google import genai

API_KEY = userdata.get("GEMINI_API_KEY")
if not API_KEY:
    raise RuntimeError("Please add your Gemini API key to Colab Secrets as GEMINI_API_KEY")

client = genai.Client(api_key=API_KEY)

def pick_generate_content_model(client):
    preferred = [
        "models/gemini-2.5-flash",
        "models/gemini-2.0-flash",
        "models/gemini-1.5-flash",
        "models/gemini-1.5-pro",
    ]

    available = []
    for m in client.models.list():
        name = getattr(m, "name", None)
        methods = getattr(m, "supported_actions", None) or getattr(m, "supported_methods", None)
        methods_str = " ".join(methods) if isinstance(methods, (list, tuple)) else str(methods)
        if name and ("generateContent" in methods_str or "generate_content" in methods_str):
            available.append(name)

    for p in preferred:
        if p in available:
            return p

    return available[0] if available else None

MODEL_NAME = pick_generate_content_model(client)
if not MODEL_NAME:
    raise RuntimeError("No available Gemini model supports generateContent for this API key.")

print("Using model:", MODEL_NAME)

def llm(prompt: str) -> str:
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )
    return (response.text or "").strip()


Using model: models/gemini-2.5-flash


## 3) Imports


In [27]:
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier
import shap

np.random.seed(42)
pd.set_option("display.max_columns", None)


## 4) Create a synthetic training dataset

We generate a dataset large enough to train a realistic model.

The label is synthetic but intentionally constructed to reflect plausible lending-like patterns:
- stronger credit and income improve approval probability
- high debt and instability reduce it
- location-based risk is included so we can discuss proxy concerns


In [28]:
n_train = 2000
train_df = pd.DataFrame({
    "annual_income_usd": np.random.randint(25000, 150000, size=n_train),
    "credit_score": np.random.randint(540, 830, size=n_train),
    "employment_years": np.random.randint(0, 20, size=n_train),
    "debt_ratio": np.round(np.random.uniform(0.03, 0.80, size=n_train), 2),
    "zip_risk_flag": np.random.choice([0, 1], size=n_train, p=[0.75, 0.25]),
    "income_volatility": np.random.choice([0, 1], size=n_train, p=[0.72, 0.28]),
})

latent = (
    (train_df["annual_income_usd"] >= 70000).astype(int) * 1.6
    + (train_df["credit_score"] >= 690).astype(int) * 2.0
    + (train_df["employment_years"] >= 4).astype(int) * 1.0
    - (train_df["debt_ratio"] >= 0.45).astype(int) * 1.6
    - (train_df["debt_ratio"] >= 0.60).astype(int) * 1.4
    - train_df["zip_risk_flag"] * 0.9
    - train_df["income_volatility"] * 0.8
    + np.random.normal(0, 0.8, size=n_train)
)

train_df["approved"] = (latent > 1.0).astype(int)
train_df.head(50)


,annual_income_usd,credit_score,employment_years,debt_ratio,zip_risk_flag,income_volatility,approved
0,146958,784,3,0.15,0,0,1
1,40795,775,8,0.39,0,0,1
2,25860,724,1,0.46,0,1,0
3,128694,811,18,0.47,0,1,1
4,144879,775,0,0.19,0,1,1
5,135268,778,6,0.60,0,1,1
6,101820,805,16,0.05,1,0,1
7,79886,604,2,0.30,0,0,1
8,31265,716,19,0.63,0,0,0
9,107386,542,19,0.46,0,0,1


## 5) Train the XGBoost model


In [ ]:
features = [
    "annual_income_usd",
    "credit_score",
    "employment_years",
    "debt_ratio",
    "zip_risk_flag",
    "income_volatility",
]

X = train_df[features]
y = train_df["approved"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

xgb_model = XGBClassifier(
    n_estimators=120,
    max_depth=4,
    learning_rate=0.08,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)

xgb_model.fit(X_train, y_train)

test_pred = xgb_model.predict(X_test)
test_proba = xgb_model.predict_proba(X_test)[:, 1]

print("Test accuracy:", round(accuracy_score(y_test, test_pred), 3))
print()
print(classification_report(y_test, test_pred))


## 6) Creating a small dataset for analysis


In [ ]:
n_demo = 50
df = pd.DataFrame({
    "applicant_id": range(1, n_demo + 1),
    "annual_income_usd": np.random.randint(25000, 130000, size=n_demo),
    "credit_score": np.random.randint(540, 810, size=n_demo),
    "employment_years": np.random.randint(0, 15, size=n_demo),
    "debt_ratio": np.round(np.random.uniform(0.05, 0.75, size=n_demo), 2),
    "zip_risk_flag": np.random.choice([0, 1], size=n_demo, p=[0.72, 0.28]),
    "income_volatility": np.random.choice([0, 1], size=n_demo, p=[0.70, 0.30]),
})
df.head(50)


## 7) Model 1 — Prediction Agent (XGBoost)


In [ ]:
df["pred_probability"] = xgb_model.predict_proba(df[features])[:, 1]
df["pred_decision"] = np.where(df["pred_probability"] >= 0.50, "APPROVE", "DENY")
df["pred_score"] = np.round((df["pred_probability"] - 0.5) * 2, 3)

df[["applicant_id", "pred_probability", "pred_decision", "pred_score"]].head(50)


## 8) Explainability with SHAP (tailored for trees)

Even if the prediction model is explainable, the system may still need interrogation and governance.

Showing how to attribute the model’s predictions to input features.

In [ ]:
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(df[features])

mean_abs_shap = np.abs(shap_values).mean(axis=0)
importance_df = pd.DataFrame({
    "feature": features,
    "mean_abs_shap": mean_abs_shap
}).sort_values("mean_abs_shap", ascending=False)

importance_df


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(importance_df["feature"], importance_df["mean_abs_shap"])
ax.set_title("XGBoost Feature Importance via Mean Absolute SHAP", fontsize=14, pad=14)
ax.set_ylabel("Mean |SHAP value|")
ax.tick_params(axis="x", rotation=30)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()


## 9) Local explanation for a single case
What influenced the prediction before the TIR agent interrogates it.


In [ ]:
case_idx = 29
local_explanation = pd.DataFrame({
    "feature": features,
    "value": df.loc[case_idx, features].values,
    "shap_value": shap_values[case_idx]
}).sort_values("shap_value", ascending=False)

local_explanation


## 10) Helper: robust JSON extraction

Making parsing safer in case that Gemini wrapped JSON in code fences.


In [16]:
def extract_json(text):
    text = text.strip()

    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)

    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1:
        raise ValueError("No JSON object found")

    return json.loads(text[start:end+1])


## 11) Model 2 — **TIR Interrogator Agent** (Gemini)

This agent does **not** make the final decision.  
Its role is to:
- interrogate the prediction
- surface proxy risks
- identify missing context
- generate structured questions


In [ ]:
def tir_interrogator_agent(row):
    payload = {
        "applicant_id": int(row["applicant_id"]),
        "annual_income_usd": int(row["annual_income_usd"]),
        "credit_score": int(row["credit_score"]),
        "employment_years": int(row["employment_years"]),
        "debt_ratio": float(row["debt_ratio"]),
        "zip_risk_flag": int(row["zip_risk_flag"]),
        "income_volatility": int(row["income_volatility"]),
        "prediction_score": float(row["pred_score"]),
        "prediction_probability": float(row["pred_probability"]),
        "prediction_decision": str(row["pred_decision"]),
    }

    prompt = f'''
You are Model 2 in a multi-agent AI system.
Your role is Interrogative Reasoning.

You do NOT make the final decision.
You question the prediction and surface governance-relevant concerns.

Case:
{json.dumps(payload, indent=2)}

Return STRICT JSON only in this exact schema:
{{
  "questions": ["string", "string"],
  "proxy_risk_detected": true,
  "uncertainty_level": "low",
  "concern_flags": ["string", "string"],
  "interrogation_summary": "string"
}}

Guidelines:
- If zip_risk_flag is 1, assess whether it may be acting as a proxy for social or economic disadvantage.
- If the prediction probability is near 0.50, consider the case borderline.
- Focus on fairness, context, missing information, proxy variables, and premature automation.
- Questions should be concise and serious.
- concern_flags should be short labels like "proxy-risk", "borderline-case", "missing-context", "high-debt-burden".
- Return valid JSON only.
'''

    raw = llm(prompt)
    try:
        return extract_json(raw)
    except Exception:
        return {
            "questions": ["Could this decision be relying on incomplete or proxy-based information?"],
            "proxy_risk_detected": bool(row["zip_risk_flag"] == 1),
            "uncertainty_level": "high",
            "concern_flags": ["parsing-fallback"],
            "interrogation_summary": raw[:500]
        }

sample_interrogation = tir_interrogator_agent(df.iloc[29])
sample_interrogation


## 12) Run Model 2 across the dataset


In [ ]:
interrogation_results = df.apply(tir_interrogator_agent, axis=1)

df["tir_questions"] = interrogation_results.apply(lambda x: x["questions"])
df["tir_proxy_risk_detected"] = interrogation_results.apply(lambda x: x["proxy_risk_detected"])
df["tir_uncertainty_level"] = interrogation_results.apply(lambda x: x["uncertainty_level"])
df["tir_concern_flags"] = interrogation_results.apply(lambda x: x["concern_flags"])
df["tir_summary"] = interrogation_results.apply(lambda x: x["interrogation_summary"])

df[[
    "applicant_id",
    "pred_decision",
    "tir_uncertainty_level",
    "tir_proxy_risk_detected",
    "tir_concern_flags",
    "tir_summary"
]].head(50)


## 13) Model 3 — Governance Agent (Gemini)

This agent sees:
- the original case
- the prediction
- the TIR interrogator's findings

It then decides whether to:
- APPROVE
- DENY
- REVIEW
- ESCALATE

This is the layer that converts questioning into **institutional action**.


In [ ]:
def governance_agent(row):
    payload = {
        "case": {
            "applicant_id": int(row["applicant_id"]),
            "annual_income_usd": int(row["annual_income_usd"]),
            "credit_score": int(row["credit_score"]),
            "employment_years": int(row["employment_years"]),
            "debt_ratio": float(row["debt_ratio"]),
            "zip_risk_flag": int(row["zip_risk_flag"]),
            "income_volatility": int(row["income_volatility"]),
        },
        "prediction_agent": {
            "pred_score": float(row["pred_score"]),
            "pred_probability": float(row["pred_probability"]),
            "pred_decision": str(row["pred_decision"]),
        },
        "tir_interrogator": {
            "questions": row["tir_questions"],
            "proxy_risk_detected": bool(row["tir_proxy_risk_detected"]),
            "uncertainty_level": str(row["tir_uncertainty_level"]),
            "concern_flags": row["tir_concern_flags"],
            "interrogation_summary": str(row["tir_summary"]),
        }
    }

    prompt = f'''
You are Model 3 in a multi-agent AI workflow.
You are the Governance Decision Agent.

You must produce ONE final action:
- APPROVE
- DENY
- REVIEW
- ESCALATE

Your job is to make a balanced, operationally realistic, accountable decision.
Do NOT send every case to REVIEW or ESCALATE.
APPROVE and DENY should remain the default for strong, low-risk cases.

Inputs:
{json.dumps(payload, indent=2)}

Return STRICT JSON only in this exact schema:
{{
  "final_action": "APPROVE",
  "confidence": "high",
  "governance_reasoning": "string",
  "requires_human_review": false
}}

Decision rules:
1. APPROVE if:
   - prediction_probability >= 0.75
   - uncertainty_level == "low"
   - proxy_risk_detected == false
   - concern_flags has at most 1 minor item

2. DENY if:
   - prediction_probability <= 0.25
   - uncertainty_level == "low"
   - proxy_risk_detected == false
   - concern_flags has at most 1 minor item

3. REVIEW if:
   - prediction_probability is between 0.25 and 0.75
   - OR uncertainty_level == "medium"
   - OR there is one meaningful concern that needs checking

4. ESCALATE only if:
   - proxy_risk_detected == true AND uncertainty_level == "high"
   - OR there are multiple serious concern_flags
   - OR the case suggests fairness, proxy, or institutional accountability risk

5. Keep the system practical:
   - APPROVE and DENY should be used when justified
   - REVIEW is for borderline or incomplete cases
   - ESCALATE is rare and reserved for high-risk cases

Return valid JSON only.
'''

    raw = llm(prompt)
    try:
        return extract_json(raw)
    except Exception:
        prob = float(row["pred_probability"])
        uncertainty = str(row["tir_uncertainty_level"])
        proxy = bool(row["tir_proxy_risk_detected"])
        concerns = row["tir_concern_flags"] if isinstance(row["tir_concern_flags"], list) else []

        if prob >= 0.75 and uncertainty == "low" and not proxy and len(concerns) <= 1:
            fallback_action = "APPROVE"
        elif prob <= 0.25 and uncertainty == "low" and not proxy and len(concerns) <= 1:
            fallback_action = "DENY"
        elif proxy and uncertainty == "high" and len(concerns) >= 2:
            fallback_action = "ESCALATE"
        else:
            fallback_action = "REVIEW"

        return {
            "final_action": fallback_action,
            "confidence": "low",
            "governance_reasoning": raw[:500],
            "requires_human_review": fallback_action in ["REVIEW", "ESCALATE"]
        }

sample_governance = governance_agent(df.iloc[29])
sample_governance

## 14) Run Model 3 across the dataset


In [ ]:
governance_results = df.apply(governance_agent, axis=1)

df["final_action"] = governance_results.apply(lambda x: x["final_action"])
df["governance_confidence"] = governance_results.apply(lambda x: x["confidence"])
df["governance_reasoning"] = governance_results.apply(lambda x: x["governance_reasoning"])
df["requires_human_review"] = governance_results.apply(lambda x: x["requires_human_review"])

df[[
    "applicant_id",
    "pred_decision",
    "pred_probability",
    "tir_uncertainty_level",
    "final_action",
    "governance_confidence",
    "requires_human_review"
]].head(50)


## 15) Compare prediction vs final governance output



In [ ]:
summary = pd.DataFrame({
    "Prediction Agent": {
        "APPROVE": int((df["pred_decision"] == "APPROVE").sum()),
        "DENY": int((df["pred_decision"] == "DENY").sum()),
        "REVIEW": 0,
        "ESCALATE": 0
    },
    "Final Governance Output": {
        "APPROVE": int((df["final_action"] == "APPROVE").sum()),
        "DENY": int((df["final_action"] == "DENY").sum()),
        "REVIEW": int((df["final_action"] == "REVIEW").sum()),
        "ESCALATE": int((df["final_action"] == "ESCALATE").sum())
    }
}).fillna(0).astype(int)

summary


## 16) Data in chart


In [ ]:
labels = ["APPROVE", "DENY", "REVIEW", "ESCALATE"]
pred_counts = [summary.loc[label, "Prediction Agent"] for label in labels]
gov_counts = [summary.loc[label, "Final Governance Output"] for label in labels]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 6))
ax.bar(x - width/2, pred_counts, width, label="Prediction Agent")
ax.bar(x + width/2, gov_counts, width, label="Final Governance Output")

ax.set_title("Multi-Agent Decision Flow: XGBoost Prediction vs Governance Outcome", fontsize=15, pad=16)
ax.set_ylabel("Number of Cases")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

for i, v in enumerate(pred_counts):
    ax.text(i - width/2, v + 0.2, str(v), ha="center", va="bottom", fontsize=10)
for i, v in enumerate(gov_counts):
    ax.text(i + width/2, v + 0.2, str(v), ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()


## 17) Single-case analysis

Walk through one example end-to-end.


In [ ]:
case = df.iloc[29]

print("Applicant ID:", int(case["applicant_id"]))
print("\n--- Model 1: XGBoost Prediction Agent ---")
print("Prediction score:", case["pred_score"])
print("Prediction probability:", round(float(case["pred_probability"]), 3))
print("Prediction decision:", case["pred_decision"])

print("\nTop local SHAP effects for this case:")
display(local_explanation)

print("\n--- Model 2: TIR Interrogator ---")
print("Uncertainty:", case["tir_uncertainty_level"])
print("Proxy risk detected:", case["tir_proxy_risk_detected"])
print("Concern flags:", case["tir_concern_flags"])
print("Questions:")
for q in case["tir_questions"]:
    print("-", q)
print("Summary:", case["tir_summary"])

print("\n--- Model 3: Governance Agent ---")
print("Final action:", case["final_action"])
print("Governance confidence:", case["governance_confidence"])
print("Requires human review:", case["requires_human_review"])
print("Reasoning:", case["governance_reasoning"])


## 18) What this demonstrates

This notebook shows a **multi-agent architecture for Interrogative Reasoning**:

- **Model 1** predicts with a real XGBoost model
- **Model 2** interrogates
- **Model 3** governs

Shows how nterrogation is operationalized as a separate layer that can influence the final outcome.

### Improvement:
Without this architecture, the system produces:
- APPROVE
- DENY

With this architecture, the system also produce:
- REVIEW
- ESCALATE

That is the meaning of **decision-space expansion** in TIR.

### Summary
Even when the prediction model is modern and explainable, the system still needs interrogation and governance to become accountable.
